# Getting started with PFLARE

This series of notebooks introduces PFLARE, a library that provides two new preconditioner (PC) types for PETSc:

- `PCPFLAREINV`: a collection of approximate inverses, which are useful as smoothers in multigrid methods or as standalone preconditioners.
- `PCAIR`: a reduction-based multigrid hierarchy using Approximate Ideal Restriction (AIR).

The notebooks are:
**[01_getting_started.ipynb](01_getting_started.ipynb)**: Introduce PFLARE
**[02_pcpflareinv.ipynb](02_pcpflareinv.ipynb)**: Examine some of the approximate inverses found in PCPFLAREINV.
**[03_cf_splitting.ipynb](03_cf_splitting.ipynb)**: Visualize the C/F splitting and explore the PMISR-DDC algorithm.
**[04_pcair.ipynb](04_pcair.ipynb)**: Introduction to PCAIR and the AIRG method.
**[05_parallel.ipynb](05_parallel.ipynb)**: Discuss PCAIR, parallelism and GPUs

**Prerequisites**: `petsc4py` and `PFLARE` must be installed and importable. While these notebooks use Python for demonstration purposes, PFLARE can be easily used in C, C++, and Fortran applications.

## 1. Import and register PFLARE

In Python, importing the `pflare` module automatically invokes `PCRegister_PFLARE()`, making the `PCAIR` and `PCPFLAREINV` types available within PETSc.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc

import pflare  # registers 'air' and 'pflareinv' PC types

print("PETSc version:", PETSc.Sys.getVersion())
print("PFLARE imported successfully")

## 2. Assemble a 2D Laplacian

We construct a standard 5-point finite-difference Laplacian on an $N \times N$ grid. This results in a symmetric positive definite (SPD) matrix, which is straightforward for most solvers to handle. We use this simple problem initially to verify that PFLARE is configured correctly.

Subsequent notebooks will explore **advection-dominated** problems, where AIR methods demonstrates their strengths.

In [ ]:
def build_2d_laplacian(N):
    """5-point FD Laplacian on an N×N grid. Returns (A, b, x)."""
    n = N * N
    A = PETSc.Mat().create()
    A.setSizes([n, n])
    A.setFromOptions()
    A.setPreallocationNNZ(5)
    A.setUp()

    h2 = 1.0 / ((N + 1) ** 2)  # (1/h)^2 factor absorbed into stencil
    rstart, rend = A.getOwnershipRange()

    for row in range(rstart, rend):
        i, j = divmod(row, N)
        A.setValue(row, row, 4.0)
        if j > 0:     A.setValue(row, row - 1, -1.0)
        if j < N - 1: A.setValue(row, row + 1, -1.0)
        if i > 0:     A.setValue(row, row - N, -1.0)
        if i < N - 1: A.setValue(row, row + N, -1.0)

    A.assemblyBegin()
    A.assemblyEnd()

    b = A.createVecRight()
    b.set(1.0)
    x = A.createVecRight()
    x.set(0.0)
    return A, b, x

N = 50  # 50×50 = 2500 unknowns
A, b, x = build_2d_laplacian(N)
print(f"Matrix size: {A.getSize()[0]} × {A.getSize()[1]}")
print(f"Number of non-zeros: {A.getInfo()['nz_used']:.0f}")

## 3. Solve with PCPFLAREINV

By default, `PCPFLAREINV` applies an assembled, sparsity-controlled GMRES polynomial to form an approximate inverse $p_k(A) \approx A^{-1}$. Several other types of approximate inverses are also available within `PCPFLAREINV`.

These approximate inverses can serve as standalone preconditioners or as effective smoothers within the `PCAIR` multigrid hierarchy.

We use GMRES as the outer Krylov solver and collect residual norms to plot convergence.

In [ ]:
def solve_and_collect(A, b, pc_type, pc_options=None, x0=None):
    """Solve A x = b with the given PC type and return residual history."""
    opts = PETSc.Options()
    if pc_options:
        for k, v in pc_options.items():
            opts[k] = v

    ksp = PETSc.KSP().create()
    ksp.setOperators(A)
    ksp.setType('gmres')
    ksp.setTolerances(rtol=1e-10, max_it=200)

    pc = ksp.getPC()
    pc.setType(pc_type)

    residuals = []
    def monitor(ksp, it, rnorm):
        residuals.append(rnorm)
    ksp.setMonitor(monitor)

    ksp.setFromOptions()

    sol = A.createVecRight()
    if x0 is not None:
        sol.copy(x0)
    ksp.solve(b, sol)

    reason = ksp.getConvergedReason()
    print(f"  PC={pc_type:12s}  iters={ksp.getIterationNumber():4d}  reason={reason}")
    return np.array(residuals)

# -- pflareinv: by default a 6th GMRES polynomial
r_pflareinv = solve_and_collect(
    A, b,
    pc_type='pflareinv',
)

## 4. Solve with PCAIR

`PCAIR` implements a reduction-based multigrid method. While we will discuss its default options in a later notebook, we enable the `-pc_air_print_stats_timings` flag here to observe the resulting hierarchy structure.

In [ ]:
# -- air: AIRG multigrid
r_air = solve_and_collect(
    A, b,
    pc_type='air',
    pc_options={
        'pc_air_print_stats_timings': True,
    }
)

## 5. Baseline: Plain GMRES (no preconditioning)

In [ ]:
r_none = solve_and_collect(A, b, pc_type='none')

## 6. Compare convergence histories

In [ ]:
# Ensure required imports are present in this cell in case earlier cells
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

# Safely retrieve histories if earlier cells didn't run
r_none = globals().get('r_none', np.array([]))
r_pflareinv = globals().get('r_pflareinv', np.array([]))
r_air = globals().get('r_air', np.array([]))

if r_none.size > 0:
    ax.semilogy(r_none / r_none[0], 'k--', label='No preconditioner')
if r_pflareinv.size > 0:
    ax.semilogy(r_pflareinv / r_pflareinv[0], 'b-o', markersize=3, label='PCPFLAREINV (6th order)')
if r_air.size > 0:
    ax.semilogy(r_air / r_air[0], 'r-s', markersize=3, label='PCAIR / AIRG')

ax.set_xlabel('GMRES iteration')
ax.set_ylabel('Relative residual')
ax.set_title(f'2D Laplacian ({N}×{N} grid, {N*N} unknowns)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('convergence_laplacian.png', dpi=120)
plt.show()
print("Saved: convergence_laplacian.png")

## 7. What's next?

AIR methods, however, are specifically designed to excel on **strongly advective** and highly asymmetric problems. To understand the mechanics of AIR methods, we will first examine the approximate inverses provided by `PCPFLAREINV`.

**[02_pcpflareinv.ipynb](02_pcpflareinv.ipynb)**: Examine some of the approximate inverses found in PCPFLAREINV.